# Mixture-of-Experts Transformer Comparison

This notebook compares the standard Probabilistic Transformer against a **Mixture-of-Experts (MoE)** architecture:

| Model | Description |
|-------|-------------|
| **Baseline (Johnson SU)** | Single Johnson SU distribution head |
| **Mixture JSU (3 components)** | Single network → mixture of 3 Johnson SU distributions (shared backbone) |
| **MoE Transformer (3 experts)** | 3 *separate* expert networks + gating network, each producing Johnson SU params |
| **MoE Transformer (5 experts)** | 5 separate expert networks + gating |

## Key difference: MoE vs Mixture head

Both produce a mixture of Johnson SU distributions at the output. The crucial difference is **architectural**:

- **Mixture JSU**: All mixture components share the same backbone up to the last layer. Specialisation is only at the parameter level.
- **MoE**: Each expert has its *own* hidden layers branching from the shared encoder. A gating network learns to route inputs to the most appropriate expert. This enforces functional specialisation — each expert learns to handle a distinct price regime (normal / spike / low-negative).

A load-balancing auxiliary loss prevents expert collapse (all weight on a single expert).

All models are run 10 times with the canonical hyperparameters to reduce statistical variability.

In [6]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from models import ProbabilisticTransformer, MoEProbabilisticTransformer
from core.experiment_utils import (
    load_data, load_cache, save_cache, run_experiment, N_RUNS,
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

GPUs detected: 1


In [7]:
RESULTS_DIR = Path(project_root) / "results" / "moe_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"

In [8]:
data = load_data()

Data loaded  —  Train: 10366, Val: 1997, Test: 3722


In [9]:
cache = load_cache(CACHE_FILE)

run_experiment(
    ProbabilisticTransformer, "Baseline (Johnson SU)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
    head_type="johnson_su",
)

run_experiment(
    ProbabilisticTransformer, "Mixture JSU (3 components)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
    head_type="mixture_johnson_su",
    head_params={"n_components": 3},
)

run_experiment(
    MoEProbabilisticTransformer, "MoE Transformer (3 experts)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
    head_params={"n_experts": 3, "balance_weight": 0.01},
)

run_experiment(
    MoEProbabilisticTransformer, "MoE Transformer (5 experts)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
    head_params={"n_experts": 5, "balance_weight": 0.01},
)

save_cache(CACHE_FILE, cache)

[Baseline (Johnson SU)] found in cache — skipping.
[Mixture JSU (3 components)] found in cache — skipping.
[MoE Transformer (3 experts)] found in cache — skipping.
[MoE Transformer (5 experts)] found in cache — skipping.


In [10]:
results = [{"Model": k, **v} for k, v in cache.items()]
df_res = pd.DataFrame(results)
if not df_res.empty:
    display(df_res.sort_values("MAE"))
    df_res.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Model,MAE,MSE,RMSE,MAPE,R2,PICP,MPIW,PINAW,IntervalScore,CRPS,NLL,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball,training_time
1,Mixture JSU (3 components),19.845403,711.626553,26.660840,1915.837883,0.494170,0.871958,81.630104,0.226620,151.283587,14.482069,NaN,5.041230,9.809403,4.618160,6.489597,100.438266
0,Baseline (Johnson SU),20.295799,736.351578,27.120937,1993.496473,0.476595,0.894619,93.245645,0.258866,151.594192,14.662948,53.875411,5.171427,9.916186,4.603888,6.563834,102.872665
3,MoE Transformer (5 experts),20.320322,739.946278,27.180528,1838.390016,0.474040,0.879888,87.227721,0.242160,154.399788,14.744320,47.487971,5.017698,9.994794,4.795551,6.602681,124.872620
2,MoE Transformer (3 experts),20.446658,749.737201,27.356310,1845.593291,0.467080,0.891294,89.695766,0.249011,150.452991,14.763834,48.594124,4.988841,10.034953,4.775970,6.599921,116.649254
